# 🌿 PlantCare AI — End-to-End GPU Training & Robustness Benchmark
### Complete Multi-Model Transfer Learning & Evaluation on the Real PlantVillage Dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shubhamydvv/PlantCare-AI/blob/main/PlantCare_AI_Colab_Training.ipynb)

---

## ⚙️ Step 1: Verify Hardware Accelerator (GPU)
> **Note:** Make sure you set your Colab runtime to GPU: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.

In [ ]:
import torch
print('=== GPU Sanity Check ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type > T4 GPU and restart.')

## 📦 Step 2: Clone Repository & Install Dependencies

In [ ]:
import os

# Clone repository if not already in directory
if not os.path.exists('src'):
    !git clone https://github.com/shubhamydvv/PlantCare-AI.git /content/PlantCare-AI
    %cd /content/PlantCare-AI

# Install project dependencies and kagglehub for automated dataset download
!pip install -q -r requirements.txt kagglehub
print('Environment and dependencies ready!')

## 📥 Step 3: Automated Dataset Download & Ingestion (PlantVillage ~54k Images)

In [ ]:
import os
import shutil
from pathlib import Path
import kagglehub

print('Downloading full PlantVillage dataset via kagglehub...')
raw_cache = kagglehub.dataset_download('emmarex/plantdisease')
print(f'Downloaded raw archive to cache: {raw_cache}')

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

cache_path = Path(raw_cache)
class_folders = [d for d in cache_path.rglob('*') if d.is_dir() and any(k in d.name for k in ['Tomato', 'Apple', 'Corn', 'Potato', 'Grape', 'Pepper'])]

if class_folders:
    source_root = class_folders[0].parent
else:
    source_root = cache_path

print(f'Ingesting classes from {source_root} into {raw_dir}...')
for cls_dir in source_root.iterdir():
    if cls_dir.is_dir() and not cls_dir.name.startswith('.'):
        dest = raw_dir / cls_dir.name
        if not dest.exists():
            shutil.copytree(cls_dir, dest)

class_count = len(list(raw_dir.iterdir()))
print(f'Successfully prepared {class_count} disease classes in {raw_dir}!')

## 🔄 Step 4: Stratified Dataset Splitting (70% Train / 15% Val / 15% Test)

In [ ]:
!python -m src.preprocessing

## 🚀 Step 5: Train Deep Learning Architectures on GPU
We train ResNet50, EfficientNet-B0, and ViT-B/16.

In [ ]:
print('=== 1. Training ResNet-50 ===')
!python -m src.train --model resnet50 --epochs 5 --batch-size 32

In [ ]:
print('=== 2. Training EfficientNet-B0 ===')
!python -m src.train --model efficientnet_b0 --epochs 5 --batch-size 32

In [ ]:
print('=== 3. Training Vision Transformer (ViT-B/16) ===')
!python -m src.train --model vit_b_16 --epochs 5 --batch-size 16 --lr 0.0001

## 📊 Step 6: Generate Robustness Benchmark & Evaluation Visualizations

In [ ]:
!python -m src.evaluate

## 📈 Step 7: Display Comparison Results & Charts

In [ ]:
import pandas as pd
from IPython.display import Image, display

metrics_csv = 'results/metrics/comparison_table.csv'
if os.path.exists(metrics_csv):
    df = pd.read_csv(metrics_csv)
    print('=== Final Model Comparison Table ===')
    display(df)

for chart_path in [
    'results/graphs/accuracy_f1_comparison.png',
    'results/graphs/latency_vs_f1_scatter.png',
    'results/graphs/reliability_calibration_diagram.png'
]:
    if os.path.exists(chart_path):
        print(f'\n--- {chart_path} ---')
        display(Image(chart_path))

## 💾 Step 8: Package & Download Results and Weights to Local PC

In [ ]:
import shutil
from google.colab import files

print('Creating zip packages...')
shutil.make_archive('plantcare_results', 'zip', 'results')
shutil.make_archive('plantcare_models', 'zip', 'models')

print('Triggering browser download for results.zip and models.zip...')
files.download('plantcare_results.zip')
files.download('plantcare_models.zip')